# 05 — Fluid Bed Coating Digital Twin

Full process chain: **Pre-heating → Spraying → Drying**.

Each stage is initialised from the end-state of the previous stage (particle
temperature, residual acetone, coating mass). Adjust the sliders to explore how
recipe and material parameters propagate through the entire coating run.

**Panels:**
1. Inlet / outlet / product temperature — full process timeline
2. Acetone on particles — full process timeline
3. Acetone in gas phase — full process timeline
4. Coating weight gain — accumulation during spraying, attrition during drying


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from fluid_bed.parameters import ProcessParameters
from fluid_bed.models.preheating  import run_preheating
from fluid_bed.models.spraying    import run_spraying
from fluid_bed.models.drying_stage import run_drying, calc_r_drying

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})


In [2]:
# Hardware / physical constants — not user-adjustable
_RHO_AIR  = 1.10    # kg/m³  (warm process air)
_CP_AIR   = 1010.0  # J/kg/K
_RHO_PART = 1050.0  # kg/m³  (typical EC-coated pellet density)
_CP_PART  = 1400.0  # J/kg/K
_D_BED    = 0.60    # m      (bed column diameter)
print("Physical constants loaded.")


Physical constants loaded.


In [ ]:
S  = dict(continuous_update=False)
S2 = dict(continuous_update=False, readout_format=".2f")

# ── Widget definitions ─────────────────────────────────────────────────────────

# --- Particle properties ---
_w = {
    # Particle
    "d_mm":          widgets.FloatSlider(1.00, min=0.80, max=1.50, step=0.05,
                         description="Diameter (mm)",   style={"description_width":"150px"}, **S),
    "ssa_cm2g":      widgets.FloatSlider(65.0, min=55.0, max=75.0, step=1.0,
                         description="SSA (cm2/g)",     style={"description_width":"150px"}, **S),
    "T0_C":          widgets.FloatSlider(20.0, min=15.0, max=30.0, step=1.0,
                         description="T0 particle (C)", style={"description_width":"150px"}, **S),
    # Batch
    "batch_kg":      widgets.FloatSlider(4.5,  min=3.0,  max=6.0,  step=0.1,
                         description="Batch size (kg)", style={"description_width":"150px"}, **S),
    "humidity_g_kg": widgets.FloatSlider(0.0,  min=0.0,  max=0.5,  step=0.05,
                         description="Humidity (g/kg)", style={"description_width":"150px"}, **S),
    "dmc_pct":       widgets.FloatSlider(1.0,  min=1.0,  max=2.0,  step=0.1,
                         description="DMC (% m/m)",     style={"description_width":"150px"}, **S),
    "coating_level": widgets.FloatSlider(0.5,  min=-1.0, max=1.0,  step=0.05,
                         description="Coating level",   style={"description_width":"150px"}, **S),
    # Preheating
    "ph_T_C":        widgets.FloatSlider(50.0, min=40.0, max=60.0, step=2.0,
                         description="[PH] T inlet (C)",    style={"description_width":"150px"}, **S),
    "ph_flow":       widgets.FloatSlider(250,  min=200,  max=400,  step=10,
                         description="[PH] Flow (m3/h)",    style={"description_width":"150px"}, **S),
    "ph_dur_min":    widgets.FloatSlider(20.0, min=10.0, max=30.0, step=1.0,
                         description="[PH] Duration (min)", style={"description_width":"150px"}, **S),
    # Spraying
    "sp_T_C":        widgets.FloatSlider(40.0, min=30.0, max=60.0, step=2.0,
                         description="[SP] T inlet (C)",    style={"description_width":"150px"}, **S),
    "sp_flow":       widgets.FloatSlider(250,  min=200,  max=400,  step=10,
                         description="[SP] Flow (m3/h)",    style={"description_width":"150px"}, **S),
    "sp_rate_g_min": widgets.FloatSlider(120,  min=95,   max=150,  step=5,
                         description="[SP] Spray (g/min)",  style={"description_width":"150px"}, **S),
    "sp_atom_bar":   widgets.FloatSlider(1.6,  min=1.2,  max=2.0,  step=0.1,
                         description="[SP] Atom. (bar)",    style={"description_width":"150px"}, **S2),
    "r_spr_1e6":     widgets.FloatSlider(6.7,  min=4.0,  max=10.0, step=0.1,
                         description="[SP] r_spray (1e-6)", style={"description_width":"150px"}, **S2),
    # Drying
    "dr_T_C":        widgets.FloatSlider(40.0, min=30.0, max=60.0, step=2.0,
                         description="[DR] T inlet (C)",    style={"description_width":"150px"}, **S),
    "dr_flow":       widgets.FloatSlider(250,  min=200,  max=400,  step=10,
                         description="[DR] Flow (m3/h)",    style={"description_width":"150px"}, **S),
    "dr_dur_min":    widgets.FloatSlider(9.0,  min=3.0,  max=15.0, step=1.0,
                         description="[DR] Duration (min)", style={"description_width":"150px"}, **S),
}

# ── Organised layout ───────────────────────────────────────────────────────────

def _lbl(text, color):
    return widgets.HTML(
        f'<b style="background:{color};color:white;padding:3px 8px;'
        f'border-radius:4px">{text}</b>')

_left_col = widgets.VBox([
    _lbl("Particle properties", "#4472C4"),
    _w["d_mm"], _w["ssa_cm2g"], _w["T0_C"],
    _lbl("Batch parameters", "#4472C4"),
    _w["batch_kg"], _w["humidity_g_kg"], _w["dmc_pct"], _w["coating_level"],
])
_right_col = widgets.VBox([
    _lbl("Pre-heating recipe", "#990536"),
    _w["ph_T_C"], _w["ph_flow"], _w["ph_dur_min"],
    _lbl("Spraying recipe", "#ED7D31"),
    _w["sp_T_C"], _w["sp_flow"], _w["sp_rate_g_min"],
    _w["sp_atom_bar"], _w["r_spr_1e6"],
    _lbl("Drying recipe", "#70AD47"),
    _w["dr_T_C"], _w["dr_flow"], _w["dr_dur_min"],
])
_panel = widgets.HBox([_left_col, _right_col],
                       layout=widgets.Layout(gap="30px"))


# ── Simulation & plotting ──────────────────────────────────────────────────────

def _run05(d_mm, ssa_cm2g, T0_C, batch_kg, humidity_g_kg, dmc_pct,
           coating_level, ph_T_C, ph_flow, ph_dur_min, sp_T_C, sp_flow,
           sp_rate_g_min, sp_atom_bar, r_spr_1e6, dr_T_C, dr_flow, dr_dur_min):

    # --- Derived quantities ---
    dmc_frac   = dmc_pct / 100.0
    humidity   = humidity_g_kg / 1000.0          # g/kg → kg/kg
    T0_K       = T0_C + 273.15

    # Spray quantity from coating level, then duration
    qty_sol_kg  = (0.0017 * coating_level + 0.0064) * batch_kg / dmc_frac
    sp_rate_kgs = sp_rate_g_min / 60.0 / 1000.0  # kg/s
    sp_dur_s    = qty_sol_kg / sp_rate_kgs

    # Mass air flows [kg/s]
    ph_m = (ph_flow / 3600.0) * _RHO_AIR
    sp_m = (sp_flow / 3600.0) * _RHO_AIR
    dr_m = (dr_flow / 3600.0) * _RHO_AIR

    # Temperatures [K]
    ph_K = ph_T_C + 273.15
    sp_K = sp_T_C + 273.15
    dr_K = dr_T_C + 273.15

    _PHYS = dict(
        diameter_eq      = d_mm * 1e-3,
        ssa_cm2_g        = ssa_cm2g,
        particle_density = _RHO_PART,
        cp_particle      = _CP_PART,
        diameter_bed     = _D_BED,
        rho_air          = _RHO_AIR,
        cp_air           = _CP_AIR,
        batch_size       = batch_kg,
    )

    # ── Stage 1: Pre-heating ────────────────────────────────────────────────
    p_ph = ProcessParameters(
        **_PHYS,
        air_flow_rates    = (ph_m, ph_m, ph_m),
        air_temperatures  = (ph_K, ph_K, ph_K),
        air_inlet_moisture= (humidity, humidity, humidity),
    )
    res_ph = run_preheating(p_ph, duration=ph_dur_min * 60.0, T_particle_init=T0_K)

    # ── Stage 2: Spraying ───────────────────────────────────────────────────
    p_sp = ProcessParameters(
        **_PHYS,
        air_flow_rates     = (sp_m, sp_m, sp_m),
        air_temperatures   = (sp_K, sp_K, sp_K),
        air_inlet_moisture = (humidity, humidity, humidity),
        spray_rate         = sp_rate_kgs,
        dry_matter_conc    = dmc_frac,
        r_spraying         = r_spr_1e6 * 1e-6,
    )
    res_sp = run_spraying(p_sp, duration=sp_dur_s,
                          T_particle_init=res_ph.T_particle[-1])

    # ── Stage 3: Drying — r_drying from DoE correlation ────────────────────
    Mc_sp_end    = res_sp.M_coating[-1]
    ec_level_pct = Mc_sp_end / batch_kg * 100.0
    try:
        r_dry = calc_r_drying(
            spray_rate_g_min    = sp_rate_g_min,
            ec_level_pct        = max(ec_level_pct, 0.01),
            batch_size_kg       = batch_kg,
            drying_time_min     = dr_dur_min,
            ec_conc_pct         = dmc_pct,
            inlet_humidity_g_kg = humidity_g_kg,
        )
    except Exception:
        r_dry = 3.19e-3

    p_dr = ProcessParameters(
        **_PHYS,
        air_flow_rates     = (dr_m, dr_m, dr_m),
        air_temperatures   = (dr_K, dr_K, dr_K),
        air_inlet_moisture = (humidity, humidity, humidity),
        r_drying           = r_dry,
    )
    res_dr = run_drying(
        p_dr,
        duration        = dr_dur_min * 60.0,
        Y_particle_init = res_sp.Y_particle[-1],
        Y_gas_init      = res_sp.Y_gas[-1],
        M_coating_init  = Mc_sp_end,
        T_particle_init = res_sp.T_particle[-1],
    )

    # ── Build continuous time axis ──────────────────────────────────────────
    t_ph = res_ph.t / 60.0
    t_sp = (res_sp.t + res_ph.t[-1])           / 60.0
    t_dr = (res_dr.t + res_ph.t[-1] + res_sp.t[-1]) / 60.0

    t_all = np.concatenate([t_ph, t_sp, t_dr])
    ph_end, sp_end = t_ph[-1], t_sp[-1]

    # Inlet temperature step function for display
    t_step = [0, ph_end, ph_end, sp_end, sp_end, t_dr[-1]]
    T_step = [ph_T_C, ph_T_C, sp_T_C, sp_T_C, dr_T_C, dr_T_C]

    # Concatenated signals
    T_prod = np.concatenate([res_ph.T_particle - 273.15,
                              res_sp.T_particle - 273.15,
                              res_dr.T_particle - 273.15])
    T_gas  = np.concatenate([res_ph.T_gas - 273.15,
                              res_sp.T_gas - 273.15,
                              res_dr.T_gas - 273.15])
    Y_part = np.concatenate([np.zeros_like(t_ph),
                              res_sp.Y_particle * 100.0,
                              res_dr.Y_particle * 100.0])
    Y_gas  = np.concatenate([np.zeros_like(t_ph),
                              res_sp.Y_gas * 100.0,
                              res_dr.Y_gas * 100.0])
    WG     = np.concatenate([np.zeros_like(t_ph),
                              res_sp.M_coating / batch_kg * 100.0,
                              res_dr.M_coating / batch_kg * 100.0])

    # ── Plotting ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(16, 9))
    fig.suptitle("Fluid Bed Coating — Digital Twin", fontsize=13, fontweight="bold")

    def _shade(ax):
        ax.axvspan(0,      ph_end, alpha=0.07, color="royalblue")
        ax.axvspan(ph_end, sp_end, alpha=0.07, color="darkorange")
        ax.axvspan(sp_end, t_dr[-1], alpha=0.07, color="seagreen")
        ax.axvline(ph_end, color="grey", lw=0.8, ls="--", alpha=0.5)
        ax.axvline(sp_end, color="grey", lw=0.8, ls="--", alpha=0.5)
        ax.set_xlabel("Time (min)")

    def _stage_labels(ax):
        ymax = ax.get_ylim()[1]
        ymin = ax.get_ylim()[0]
        ypos = ymin + (ymax - ymin) * 0.96
        ax.text((0 + ph_end)/2,   ypos, "PH", ha="center", va="top",
                fontsize=8, color="royalblue", fontweight="bold")
        ax.text((ph_end+sp_end)/2, ypos, "SP", ha="center", va="top",
                fontsize=8, color="darkorange", fontweight="bold")
        ax.text((sp_end+t_dr[-1])/2, ypos, "DR", ha="center", va="top",
                fontsize=8, color="seagreen", fontweight="bold")

    # Panel 1 — Temperatures
    ax = axes[0, 0]
    _shade(ax)
    ax.step(t_step, T_step, color="tomato",   lw=1.5, ls="--", where="post", label="T inlet")
    ax.plot(t_all,  T_gas,  color="tomato",   lw=1.5, alpha=0.55,           label="T outlet (gas)")
    ax.plot(t_all,  T_prod, color="steelblue",lw=2.5,                       label="T product")
    ax.set_ylabel("Temperature (°C)")
    ax.set_title("Temperature profiles")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    _stage_labels(ax)

    # Panel 2 — Acetone on particles
    ax = axes[0, 1]
    _shade(ax)
    ax.plot(t_all, Y_part, color="darkorange", lw=2.5)
    ax.set_ylabel("Acetone on particles (wt %)")
    ax.set_title("Particle acetone content"); ax.grid(True, alpha=0.3)
    _stage_labels(ax)

    # Panel 3 — Acetone in gas
    ax = axes[1, 0]
    _shade(ax)
    ax.plot(t_all, Y_gas, color="cadetblue", lw=2.5)
    ax.set_ylabel("Acetone in gas phase (wt %)")
    ax.set_title("Gas-phase acetone"); ax.grid(True, alpha=0.3)
    _stage_labels(ax)

    # Panel 4 — Coating weight gain
    ax = axes[1, 1]
    _shade(ax)
    ax.plot(t_all, WG, color="mediumpurple", lw=2.5)
    ax.set_ylabel("Coating weight gain (%)")
    ax.set_title("Coating deposition & attrition"); ax.grid(True, alpha=0.3)
    _stage_labels(ax)

    plt.tight_layout()
    plt.show()
    plt.close()

    # ── Summary ─────────────────────────────────────────────────────────────
    wg_sp  = res_sp.M_coating[-1] / batch_kg * 100.0
    wg_fin = res_dr.M_coating[-1] / batch_kg * 100.0
    print(
        f"Spray qty    : {qty_sol_kg*1000:.1f} g  |  "
        f"Spray dur    : {sp_dur_s/60:.1f} min  |  "
        f"Total process: {t_dr[-1]:.1f} min"
    )
    print(
        f"WG after SP  : {wg_sp:.3f}%  |  "
        f"WG final (DR): {wg_fin:.3f}%  |  "
        f"r_drying     : {r_dry:.3e} kg/s  (from DoE correlation)"
    )


_out05 = widgets.interactive_output(_run05, _w)
clear_output(wait=True)
display(_panel, _out05)


Output()